<h1 style="color: #119d5c;">Step 1 of the Eutrophication workbench workflow:</h1>
<h2 style="color: #112d8c;">EWB BEACON Query Notebook: Input Generator for the CW Duplicate Tool </h2>

Welcome! 👋  
This notebook is designed to be a **simple, flexible starting point** for exploring, filter the data and running **Step 1** of the workflow.
It allows you to query **harmonized Essential Ocean Variables (EOV) data** from the merged **BEACON** instance.  

The output of this notebook is a **Parquet file** filtered by BDI, harmonized units and feature_type that meets the minimum input requirements for **Step 2: the CW Duplicate Tool**. 

### What you can do with this notebook
- Query the **Merged BEACON instance**
- Filter data by **BDI**
- Group data by feature type: **profiles** or **time series**
- Select time range, depth range, and geographical area
- Choose among different **unit types**
- Export the results in **Parquet format**, ready for the CW tool

---

### Instructions

1. Execute the Python cells below one by one. 
2. Enter the required inputs when prompted (BDI, Feature Type, units).
3. The script will create a parquet file each time you run this notebook (1 per the selected BDI). Keep in mind you **MUST** run the notebook each time you want to retreive data from the select the BDI. 
4. At the end you should have 3 parquet files (1 per BDI) in the `/output/in_raw/` folder with the data ready to perform the duplicate analysis
---

### Access limitations
If you do not writen have access to `/blue-cloud-dataspace`, it means you are not part of the developer team.  
In that case, the **size of the datasets you can process will be limited**.

---

## Authorship

* **Author:** Nydia Catalina Reyes Suarez ([![ORCID](https://orcid.org/sites/default/files/images/orcid_16x16.png)](https://orcid.org/0000-0002-3906-471X) [ORCID](https://orcid.org/0000-0002-3906-471X), [Github](https://github.com/Geeokta))
* **Affiliation:** Istituto Nazionale di Oceanografia e di Geofisica Sperimentale - OGS  
* **Contact:** nreyessuarez@ogs.it  
* **Release date:** 2026-04-30

---

## License
This notebook is licensed under the [Creative Commons Attribution 4.0 International License](https://creativecommons.org/licenses/by/4.0/).

© 2026 NYDIA CATALINA REYES SUAREZ. All rights reserved.

---
## How to Cite:
Reyes Suarez, Nydia Catalina (2026): Jupyter Notebook - EWB BEACON Query Notebook: Input Generator for the CW Duplicate Tool. v.1.0.0. DOI: 10.5281/zenodo.19925412. Retrieved from the Blue-Cloud Gateway (https://blue-cloud.d4science.org), Eutrophication Workbench VLab (https://blue-cloud.d4science.org/group/eutrophication-workbench) operated by D4Science.org (www.d4science.org - ROR https://ror.org/027cvs066) 

---


### 1) Install the `beacon_api` package

To interact with the BEACON Data Lake API, you need the `beacon_api` Python package.

- PyPI: https://pypi.org/project/beacon-api/  
- GitHub repository: https://github.com/maris-development/beacon  
- Documentation: https://maris-development.github.io/beacon/docs/1.5.4/introduction.html

In [ ]:
%pip install beacon_api --upgrade
%pip install contextily

from beacon_api import * # Import the Beacon API client

### 2) Set your BEACON Blue‑Cloud token and check the BEACON version

To access the BEACON endpoint, you must provide your personal Blue‑Cloud token.  
You can retrieve it from the **Eutrophication Workbench home page**:

1. Go to the workbench home page  
2. In the top menu, click **How-to** and then select **Authorization How-to**
3. Click **“Get Token”** to generate your 24‑hour token  


---

#### 🔐 Token validity

- Tokens generated in **D4Science** are valid for **24 hours**  
- After expiration, you must generate a **new token**  
- When you obtain a new token, you **must restart your JupyterLab session**  
  so the updated token is correctly loaded  

The code below automatically retrieves your active token — no need to paste it manually.

> ⚠️ If you are running the notebook *outside* the D4Science VRE,  
> you must obtain the token from the DDAS portal:  
> https://data.blue-cloud.org/search  
> and insert it manually.


In [ ]:
# Set your Beacon Blue Cloud Token
TOKEN = os.getenv('D4SCIENCE_TOKEN')
merged_client = Client('https://beacon-wb2-eutrophication.d4science.org', jwt_token=TOKEN)

### List available columns and their data types

Use the variable `search_term` to filter column names.  
Example: set `search_term = "oxygen"` to find all oxygen‑related fields.

In [ ]:
# search for a specific column
columns = merged_client.available_columns_with_data_type()
search_term = "".lower()  # Convert to lowercase for case-insensitive search
[field for field in columns if search_term in field.name.lower()]

### 3) Using the Query Builder to dynamically create queries

### 3.1) BDI FILTER: Select the BDI from the merged instance

Only the **merged** option is available in this notebook.  
This allows you to filter by BDI + Featuretype and generate the **Parquet files required by the CW Duplicate Tool**.

If you need to work with individual BDI versions, refer to:

- the **monolith** Jupyter Notebooks, or  
- **`EWB_BEACON=merged_v1.5.4_var=allEOV_out=multi.ipynb`**

In [ ]:
import ipywidgets as widgets
source_bdi_widget = widgets.Dropdown(
  options=[
    ("EMODNET CHEMISTRY", "BEACON_EMODNET_CHEMISTRY"),
    ("CMEMS BGC", "BEACON_CMEMS_BGC"),
    ("WOD", "BEACON_WOD")
  ],
  value="BEACON_EMODNET_CHEMISTRY",
  description="Source BDI:"
)

display(source_bdi_widget)


In [ ]:
print(source_bdi_widget.value)

#### 3.2) Select the unit type for the dataset

Choose whether you want variables expressed **per volume** or **per mass**.  
This selection dynamically updates the column names used in the query.

In [ ]:
unit_widget = widgets.Dropdown(
    options=[
        ("PER_VOLUME", "PER_VOLUME"),
        ("PER_MASS", "PER_MASS")],
    value="PER_VOLUME",
    description="Unit type:"
)

display(unit_widget)

In [ ]:
unit = unit_widget.value
print(unit)

# Define dynamic column names based on unit
chl_col = f"CHLOROPHYLL_{unit}" if unit == "PER_VOLUME" else f"CHLOROPHYLL_PER_VOLUME" #CLOROPHYLL PER VOLUME is the only column that does not follow the new naming convention, so we need to handle it separately
oxy_col = f"OXYGEN_{unit}"
nit_col = f"NITRATE_{unit}"
nit_nit_col = f"NITRATE_NITRITE_{unit}"
amm_col = f"AMMONIUM_{unit}"
pho_col = f"PHOSPHATE_{unit}"
sil_col = f"SILICATE_{unit}"

print(chl_col, oxy_col, nit_col, nit_nit_col, amm_col, pho_col, sil_col)


### 3.3) Select the feature type: Profiles or Time Series

This determines how the output will be grouped:

- **PROFILE** → one file per profile (including profiles, timeSeriesProfile, trajectoryProfile)  
- **TIMESERIES** → one file per time series (including timeseries and trajectory)

In [ ]:
featuretype_widget = widgets.Dropdown(
    options=[
        ("PROFILE", "PROFILE"),
        ("TIMESERIES", "TIMESERIES")],
    value="PROFILE",
    description="FEATURE TYPE:"
)

display(featuretype_widget)

In [ ]:
print(featuretype_widget.value)

### 3.4) Build the query

The following cell:

- stores widget selections  
- initializes a new query  
- selects all required variables  
- adds metadata fields  
- applies filters  
- groups by feature type  
- restricts time, depth, and geographical area  

This query produces the harmonized dataset required for the CW Duplicate Tool.

In [ ]:
# Store widget values for use in output filename
source_bdi = source_bdi_widget.value
feature_type = featuretype_widget.value


print(f"source_bdi: {source_bdi}")
print(f"feature_type: {feature_type}")
print(f"unit_type: {unit}")

query = merged_client.query()  # Create a new query builder instance
query.add_select_column("COMMON_TIME", alias="TIME")
query.add_select_column("COMMON_LATITUDE", alias="LATITUDE")
query.add_select_column("COMMON_LONGITUDE", alias="LONGITUDE")

#DEPTH
query.add_select_column("COMMON_DEPTH", alias="DEPTH")
query.add_select_column("COMMON_DEPTH_QC", alias="DEPTH_QC")
query.add_select_column("COMMON_DEPTH_UNITS", alias="DEPTH_UNITS")
query.add_select_column("COMMON_DEPTH_P01", alias="DEPTH_P01")
query.add_select_column("COMMON_DEPTH_P06", alias="DEPTH_P06")

# CHLOROPHYLL
query.add_select_column(f"COMMON_{chl_col}", alias= chl_col)
query.add_select_column(f"COMMON_{chl_col}_QC", alias=chl_col + "_QC")
query.add_select_column(f"COMMON_{chl_col}_UNITS", alias=chl_col + "_UNITS")
query.add_select_column(f"COMMON_{chl_col}_P01", alias=chl_col + "_P01")
query.add_select_column(f"COMMON_{chl_col}_P06", alias=chl_col + "_P06")
query.add_select_column("COMMON_CHLOROPHYLL_L05", alias="CHLOROPHYLL_L05")
query.add_select_column("COMMON_CHLOROPHYLL_L22", alias="CHLOROPHYLL_L22")
query.add_select_column("COMMON_CHLOROPHYLL_L33", alias="CHLOROPHYLL_L33")

#OXYGEN PER VOLUME
query.add_select_column(f"COMMON_OXYGEN_{unit}", alias= oxy_col)
query.add_select_column(f"COMMON_OXYGEN_{unit}_QC", alias=oxy_col + "_QC")
query.add_select_column(f"COMMON_OXYGEN_{unit}_UNITS", alias=oxy_col + "_UNITS")
query.add_select_column(f"COMMON_OXYGEN_{unit}_P01", alias=oxy_col + "_P01")
query.add_select_column(f"COMMON_OXYGEN_{unit}_P06", alias=oxy_col + "_P06")
query.add_select_column("COMMON_OXYGEN_L05", alias="OXYGEN_L05")
query.add_select_column("COMMON_OXYGEN_L22", alias="OXYGEN_L22")
query.add_select_column("COMMON_OXYGEN_L33", alias="OXYGEN_L33")

# NITRATE PER VOLUME
query.add_select_column(f"COMMON_NITRATE_{unit}", alias= nit_col)
query.add_select_column(f"COMMON_NITRATE_{unit}_QC", alias=nit_col + "_QC")
query.add_select_column(f"COMMON_NITRATE_{unit}_UNITS", alias=nit_col + "_UNITS")
query.add_select_column(f"COMMON_NITRATE_{unit}_P01", alias=nit_col + "_P01")
query.add_select_column(f"COMMON_NITRATE_{unit}_P06", alias=nit_col + "_P06")
query.add_select_column("COMMON_NITRATE_L05", alias="NITRATE_L05")
query.add_select_column("COMMON_NITRATE_L22", alias="NITRATE_L22")
query.add_select_column("COMMON_NITRATE_L33", alias="NITRATE_L33")

# NITRATE PLUS NITRITE PER VOLUME
query.add_select_column(f"COMMON_NITRATE_NITRITE_{unit}", alias=nit_nit_col)
query.add_select_column(f"COMMON_NITRATE_NITRITE_{unit}_QC", alias=nit_nit_col + "_QC")
query.add_select_column(f"COMMON_NITRATE_NITRITE_{unit}_UNITS", alias=nit_nit_col + "_UNITS")
query.add_select_column(f"COMMON_NITRATE_NITRITE_{unit}_P01", alias=nit_nit_col + "_P01")
query.add_select_column(f"COMMON_NITRATE_NITRITE_{unit}_P06", alias=nit_nit_col + "_P06")
query.add_select_column("COMMON_NITRATE_NITRITE_L05", alias="NITRATE_NITRITE_L05")
query.add_select_column("COMMON_NITRATE_NITRITE_L22", alias="NITRATE_NITRITE_L22")
query.add_select_column("COMMON_NITRATE_NITRITE_L33", alias="NITRATE_NITRITE_L33")

# AMMONIUM PER VOLUME
query.add_select_column(f"COMMON_AMMONIUM_{unit}", alias= amm_col)
query.add_select_column(f"COMMON_AMMONIUM_{unit}_QC", alias=amm_col + "_QC")
query.add_select_column(f"COMMON_AMMONIUM_{unit}_UNITS", alias=amm_col + "_UNITS")
query.add_select_column(f"COMMON_AMMONIUM_{unit}_P01", alias=amm_col + "_P01")
query.add_select_column(f"COMMON_AMMONIUM_{unit}_P06", alias=amm_col + "_P06")
query.add_select_column("COMMON_AMMONIUM_L05", alias="AMMONIUM_L05")
query.add_select_column("COMMON_AMMONIUM_L22", alias="AMMONIUM_L22")
query.add_select_column("COMMON_AMMONIUM_L33", alias="AMMONIUM_L33")

# PHOSPHATE PER VOLUME
query.add_select_column(f"COMMON_PHOSPHATE_{unit}", alias= pho_col)
query.add_select_column(f"COMMON_PHOSPHATE_{unit}_QC", alias=pho_col + "_QC")
query.add_select_column(f"COMMON_PHOSPHATE_{unit}_UNITS", alias=pho_col + "_UNITS")
query.add_select_column(f"COMMON_PHOSPHATE_{unit}_P01", alias=pho_col + "_P01")
query.add_select_column(f"COMMON_PHOSPHATE_{unit}_P06", alias=pho_col + "_P06")
query.add_select_column("COMMON_PHOSPHATE_L05", alias="PHOSPHATE_L05")
query.add_select_column("COMMON_PHOSPHATE_L22", alias="PHOSPHATE_L22")
query.add_select_column("COMMON_PHOSPHATE_L33", alias="PHOSPHATE_L33")

# SILICATE PER VOLUME
query.add_select_column(f"COMMON_SILICATE_{unit}", alias= sil_col)
query.add_select_column(f"COMMON_SILICATE_{unit}_QC", alias=sil_col + "_QC")
query.add_select_column(f"COMMON_SILICATE_{unit}_UNITS", alias=sil_col + "_UNITS")
query.add_select_column(f"COMMON_SILICATE_{unit}_P01", alias=sil_col + "_P01")
query.add_select_column(f"COMMON_SILICATE_{unit}_P06", alias=sil_col + "_P06")
query.add_select_column("COMMON_SILICATE_L05", alias="SILICATE_L05")
query.add_select_column("COMMON_SILICATE_L22", alias="SILICATE_L22")
query.add_select_column("COMMON_SILICATE_L33", alias="SILICATE_L33")

# SALINITY
query.add_select_column("COMMON_SALINITY", alias="SALINITY")
query.add_select_column("COMMON_SALINITY_QC", alias="SALINITY_QC")
query.add_select_column("COMMON_SALINITY_UNITS", alias="SALINITY_UNITS")
query.add_select_column("COMMON_SALINITY_P01", alias="SALINITY_P01")
query.add_select_column("COMMON_SALINITY_P06", alias="SALINITY_P06")
query.add_select_column("COMMON_SALINITY_L05", alias="SALINITY_L05")
query.add_select_column("COMMON_SALINITY_L22", alias="SALINITY_L22")
query.add_select_column("COMMON_SALINITY_L33", alias="SALINITY_L33")

# TEMPERATURE
query.add_select_column("COMMON_TEMPERATURE", alias="TEMPERATURE")
query.add_select_column("COMMON_TEMPERATURE_QC", alias="TEMPERATURE_QC")
query.add_select_column("COMMON_TEMPERATURE_UNITS", alias="TEMPERATURE_UNITS")
query.add_select_column("COMMON_TEMPERATURE_P01", alias="TEMPERATURE_P01")
query.add_select_column("COMMON_TEMPERATURE_P06", alias="TEMPERATURE_P06")
query.add_select_column("COMMON_TEMPERATURE_L05", alias="TEMPERATURE_L05")
query.add_select_column("COMMON_TEMPERATURE_L22", alias="TEMPERATURE_L22")
query.add_select_column("COMMON_TEMPERATURE_L33", alias="TEMPERATURE_L33")

# add metadata columns
query.add_select_column("COMMON_PLATFORM_L06", alias="PLATFORM_L06")
query.add_select_column("COMMON_PLATFORM_C17", alias="PLATFORM_C17")
query.add_select_column("SOURCE_BDI")
query.add_select_column("SOURCE_BDI_DATASET_ID")
query.add_select_column("COMMON_EDMO_CODE", alias="EDMO_CODE")
query.add_select_column("COMMON_CSR", alias="CSR")

query.add_select_column("COMMON_HARMONIZATION_DATE")
query.add_select_column("COMMON_BDI_SNAPSHOT_DATE")
query.add_select_column("COMMON_BDI_MONOLITH_VERSION")

#metadata from monoliths
query.add_select_column(".bigram", alias="BIGRAM")
query.add_select_column("dataset", alias="WOD_DATASET")
query.add_select_column("Measuring area type", alias="MEASURING_AREA_TYPE")
query.add_select_column("COMMON_FEATURE_TYPE", alias="FEATURE_TYPE")
# important to generate the odv format
query.add_select_column("COMMON_ODV_TAG", alias="ODV_TAG")

# Apply filters to the query. WE KEEP ONLY SAMPLES WITH ASSOCIATED TEMPERATURE AND SALINITY MEASUREMENTS
query.add_filter(
        OrFilter([IsNotNullFilter(chl_col), 
                  IsNotNullFilter(oxy_col), 
                  IsNotNullFilter(nit_col),
                  IsNotNullFilter(nit_nit_col), 
                  IsNotNullFilter(amm_col),
                  IsNotNullFilter(pho_col),
                  IsNotNullFilter(sil_col),
                  ])
    )
# group by Profiles or timeseries depending on the feature type widget selection
# This will give us one file per profile or time series per BDI to use in CW configuration file during the duplicate detection step.
if featuretype_widget.value == "PROFILE":
    query.add_filter(
        OrFilter([EqualsFilter("FEATURE_TYPE", "profile"),
                  EqualsFilter("FEATURE_TYPE", "timeSeriesProfile"),
                  EqualsFilter("FEATURE_TYPE", "trajectoryProfile")
                  ])
    )
elif featuretype_widget.value == "TIMESERIES":
    query.add_filter(
        OrFilter([EqualsFilter("FEATURE_TYPE", "timeSeries"), 
                  EqualsFilter("FEATURE_TYPE", "trajectory")
                  ])  
    )


# query.add_range_filter("TIME", "1921-10-15T00:00:00", "2023-12-31T23:59:59") # full time range
query.add_range_filter("TIME", "2010-01-01T00:00:00", "2010-01-31T23:59:59") # You can adjust the date range as needed. The format is ISO 8601.

# Depth range from 0 to 10000 meters (you can adjust as needed)
query.add_range_filter("DEPTH", 0, 10) 

# comment when only querying merged data, the filter give a file per BDI
query.add_equals_filter("SOURCE_BDI", source_bdi_widget.value)

# Alternatively, you can use a polygon filter to define a custom area. Below the polygon represents the North-East Atlantic area:
# query.add_polygon_filter("LONGITUDE", "LATITUDE", [[-42, 24.30], [-42, 48], [-0.5, 48], [-0.5, 41], [-5,37], [-5, 24.30], [-42, 24.30]])

query.add_range_filter("LATITUDE", -90, 90) # Latitude range from -90 to 90 for full range (you can adjust as needed)
query.add_range_filter("LONGITUDE", -180, 180) # Longitude range from -180 to 180 for full range (you can adjust as needed)

### 4) Select the output type

The CW Duplicate Tool requires **Parquet** format.  
The query will be executed and the resulting file saved accordingly using the naming conventions defined within the project.

In [ ]:
from datetime import datetime
import pandas as pd

# First, get the dataframe to extract min/max values for naming
df = query.to_pandas_dataframe()
# Extract min/max DEPTH and TIME from the actual data
min_depth = int(df['DEPTH'].min())
max_depth = int(df['DEPTH'].max())
min_time = pd.to_datetime(df['TIME'].min()).strftime("%Y%m%d")
max_time = pd.to_datetime(df['TIME'].max()).strftime("%Y%m%d")

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Construct filename following the naming convention: 
# <PROJECT>_<WORKBENCH>_<BEACON INSTANCE>_<SOURCE BDI>_<featuretype>_<UNIT>_<minDEPTH>_to_<maxDEPTH>_<minTIME>_to_<maxTIME>_<version TIMESTAMP<.Format
filename = f"BC_EWB_MERGED_{source_bdi}_{feature_type}_{unit}_{min_depth}m_to_{max_depth}m_{min_time}_to_{max_time}_v{run_timestamp}"
query.to_parquet(f"{filename}.parquet")


In [ ]:
from pathlib import Path
from shutil import copy2, move

output_root = Path("output")
in_raw = output_root / "in_raw"

folder_map = {
    "BEACON_CMEMS_BGC": "EUTRO_CMEMS",
    "BEACON_EMODNET_CHEMISTRY": "EUTRO_EC",
    "BEACON_WOD": "EUTRO_WOD",
}

for subfolder in folder_map.values():
    (in_raw / subfolder).mkdir(parents=True, exist_ok=True)

dst_sub = folder_map.get(source_bdi)
if dst_sub is None:
    raise ValueError(f"Unknown source_bdi: {source_bdi}")

dst_dir = in_raw / dst_sub
move(f"{filename}.parquet", dst_dir / f"{filename}.parquet")